In [29]:
SESSION_ID = 2026

In [31]:
import time
import joblib

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    cohen_kappa_score,
    matthews_corrcoef,
)


In [ ]:
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score


In [ ]:
def eval_one(y_true, y_pred, y_proba=None, classes=None, fit_time_sec=None):
    y_true = pd.Series(y_true).astype(str)
    y_pred = pd.Series(y_pred).astype(str)

    
    out = {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Recall": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "Prec.": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "Kappa": float(cohen_kappa_score(y_true, y_pred)),
        "MCC": float(matthews_corrcoef(y_true, y_pred)),
        "TT (Sec)": float(fit_time_sec) if fit_time_sec is not None else np.nan,
        "bal_acc": float(balanced_accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

   
    auc = np.nan
    if y_proba is not None and classes is not None:
        classes = [str(c) for c in classes]
        try:
            if len(classes) == 2:
               
                auc = roc_auc_score((y_true == classes[1]).astype(int), y_proba[:, 1])
            else:
                auc = roc_auc_score(
                    y_true,
                    y_proba,
                    multi_class="ovr",
                    average="macro",
                    labels=classes,
                )
        except Exception:
            auc = np.nan

    out["AUC"] = float(auc) if auc == auc else np.nan  
    return out


In [ ]:
split_dir=Path("/home/chenliqun/tcga_pycaret/mincancer1_pycaret_tabpfn_splits")
fold_dir=Path("/home/chenliqun/tcga_pycaret/mincancer1_pycaret_tabpfn_splits/cv5")
out_dir=Path("./mincancer1_Final_tabpfn2.5_cv5_out")
out_dir.mkdir(parents=True,exist_ok=True)

def read_xy_from_splits(split_dir:Path):
    X_train=pd.read_csv(split_dir/"X_train.tsv",sep="\t",index_col=0)
    X_test=pd.read_csv(split_dir/"X_test.tsv",sep="\t",index_col=0)

    y_train_df=pd.read_csv(split_dir/"y_train.tsv",sep="\t",index_col=0)
    y_test_df=pd.read_csv(split_dir/"y_test.tsv",sep="\t",index_col=0)

    y_train=(y_train_df["target"] if "target" in y_train_df.columns else y_train_df.iloc[:,0]).astype(str)
    y_test=(y_test_df["target"] if "target" in y_test_df.columns else y_test_df.iloc[:,0]).astype(str)

    # Check sample alignment
    assert X_train.index.equals(y_train.index),"X_train and y_train index/order mismatch"
    assert X_test.index.equals(y_test.index),"X_test and y_test index/order mismatch"

    # Align feature columns (use intersection)
    common_cols=X_train.columns.intersection(X_test.columns)
    X_train=X_train[common_cols].copy()
    X_test=X_test[common_cols].copy()

    # Remove columns that are all NaN in train (drop in test as well)
    all_nan_cols=X_train.columns[X_train.isna().all(axis=0)]
    if len(all_nan_cols)>0:
        X_train=X_train.drop(columns=all_nan_cols)
        X_test=X_test.drop(columns=all_nan_cols)

    # Ensure index type consistency (avoid "001" vs 1 issues)
    X_train.index=X_train.index.astype(str)
    X_test.index=X_test.index.astype(str)
    y_train.index=y_train.index.astype(str)
    y_test.index=y_test.index.astype(str)

    return X_train,y_train,X_test,y_test

def read_fold_ids(fold_dir:Path):
    train_files=sorted(fold_dir.glob("fold*_train_idx.csv"))
    valid_files=sorted(fold_dir.glob("fold*_valid_idx.csv"))
    assert len(train_files)==len(valid_files) and len(train_files)>0,"Fold files mismatch or empty"

    def read_ids(p:Path):
        return pd.read_csv(p,dtype=str).iloc[:,0].astype(str).tolist()

    folds=[]
    for k in range(1,len(train_files)+1):
        tr_ids=read_ids(fold_dir/f"fold{k}_train_idx.csv")
        va_ids=read_ids(fold_dir/f"fold{k}_valid_idx.csv")
        folds.append((tr_ids,va_ids))
    return folds

def assert_folds_ok(X_train:pd.DataFrame,folds):
    idx_set=set(X_train.index)
    for i,(tr_ids,va_ids) in enumerate(folds,start=1):
        tr_set,va_set=set(tr_ids),set(va_ids)
        assert tr_set.isdisjoint(va_set),f"fold{i}: train/valid have overlapping samples"
        miss_tr=tr_set-idx_set
        miss_va=va_set-idx_set
        assert not miss_tr and not miss_va,(
            f"fold{i}: fold indices contain samples not in X_train.index.\n"
            f"missing train (show 5): {list(miss_tr)[:5]}\n"
            f"missing valid (show 5): {list(miss_va)[:5]}"
        )

# Main workflow
X_train,y_train,X_test,y_test=read_xy_from_splits(split_dir)
folds=read_fold_ids(fold_dir)
assert_folds_ok(X_train,folds)

print("Train:",X_train.shape,"n_classes:",y_train.nunique())
print("Test :",X_test.shape,"n_classes:",y_test.nunique())
print("Folds:",len(folds))

from tabpfn import TabPFNClassifier

cv_rows=[]
for fold_id,(tr_ids,va_ids) in enumerate(folds,start=1):
    X_tr,y_tr=X_train.loc[tr_ids],y_train.loc[tr_ids]
    X_va,y_va=X_train.loc[va_ids],y_train.loc[va_ids]

    model=Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("clf",TabPFNClassifier(
            model_path="/home/chenliqun/Tcga/model/tabpfn-v2.5-classifier-v2.5_default.ckpt",
            random_state=SESSION_ID
        )),
    ])

    t0=time.time()
    model.fit(X_tr,y_tr)
    fit_time=time.time()-t0

    pred_va=model.predict(X_va)
    proba_va=model.predict_proba(X_va)
    classes=model.named_steps["clf"].classes_

    m=eval_one(y_va,pred_va,y_proba=proba_va,classes=classes,fit_time_sec=fit_time)
    m["fold"]=fold_id
    m["n_train"]=int(X_tr.shape[0])
    m["n_valid"]=int(X_va.shape[0])
    cv_rows.append(m)

    # Save validation predictions for each fold (including probabilities)
    df_out=pd.DataFrame(proba_va,columns=[f"proba_{c}" for c in classes],index=X_va.index)
    df_out.insert(0,"y_true",y_va.astype(str).values)
    df_out.insert(1,"y_pred",pd.Series(pred_va,index=X_va.index).astype(str).values)
    df_out.to_csv(out_dir/f"valid_pred_fold{fold_id}.tsv",sep="\t")

cv_df=pd.DataFrame(cv_rows)

# Output metrics in specified order
metric_cols=["Accuracy","AUC","Recall","Prec.","F1","Kappa","MCC","TT (Sec)","bal_acc","f1_macro"]
cv_df=cv_df[["fold","n_train","n_valid"]+metric_cols]

cv_df.to_csv(out_dir/"cv_metrics.csv",index=False,encoding="utf-8-sig")
print("CV metrics:\n",cv_df)
print("CV mean:\n",cv_df[metric_cols].mean(numeric_only=True))

Train: (2367, 144) n_classes: 7
Test : (592, 144) n_classes: 7
Folds: 5
CV metrics:
    fold  n_train  n_valid  Accuracy       AUC    Recall     Prec.        F1  \
0     1     1893      474  0.974684  0.998858  0.974684  0.975502  0.974962   
1     2     1893      474  0.972574  0.999124  0.972574  0.974090  0.972913   
2     3     1894      473  0.978858  0.998796  0.978858  0.979555  0.978935   
3     4     1894      473  0.974630  0.999515  0.974630  0.974988  0.974411   
4     5     1894      473  0.983087  0.999349  0.983087  0.983429  0.983162   

      Kappa       MCC  TT (Sec)   bal_acc  f1_macro  
0  0.967654  0.967682  0.833083  0.948658  0.941694  
1  0.965018  0.965096  0.788738  0.967012  0.951372  
2  0.972970  0.973026  0.773697  0.974692  0.963906  
3  0.967480  0.967530  0.775530  0.938879  0.951002  
4  0.978360  0.978382  0.774577  0.976576  0.974107  
CV mean:
 Accuracy    0.976766
AUC         0.999128
Recall      0.976766
Prec.       0.977513
F1          0.976877
K

In [ ]:
# Train on full train set -> evaluate on fixed test set
final_model=Pipeline([
("imputer",SimpleImputer(strategy="median")),
("clf",TabPFNClassifier(
model_path="/home/chenliqun/Tcga/model/tabpfn-v2.5-classifier-v2.5_default.ckpt",
random_state=SESSION_ID
)),
])

t0=time.time()
final_model.fit(X_train,y_train)
final_fit_time=time.time()-t0

pred_test=final_model.predict(X_test)
proba_test=final_model.predict_proba(X_test)
classes=final_model.named_steps["clf"].classes_

# Test metrics (including training time)
test_metrics=eval_one(y_test,pred_test,y_proba=proba_test,classes=classes,fit_time_sec=final_fit_time)

test_df=pd.DataFrame([test_metrics])[metric_cols]
test_df.to_csv(out_dir/"test_metrics.csv",index=False,encoding="utf-8-sig")

print("Test metrics:\n",test_df)

# Save model (Pipeline: imputer + TabPFNClassifier)
joblib.dump(final_model,out_dir/"final_model.joblib",compress=3)
print("Saved model to:",out_dir/"final_model.joblib")

# Optionally save train/test predictions
pred_train=final_model.predict(X_train)
proba_train=final_model.predict_proba(X_train)

train_out=pd.DataFrame(proba_train,columns=[f"proba_{c}" for c in classes],index=X_train.index)
train_out.insert(0,"y_true",y_train.astype(str).values)
train_out.insert(1,"y_pred",pd.Series(pred_train,index=X_train.index).astype(str).values)
train_out.to_csv(out_dir/"train_pred.tsv",sep="\t")

test_out=pd.DataFrame(proba_test,columns=[f"proba_{c}" for c in classes],index=X_test.index)
test_out.insert(0,"y_true",y_test.astype(str).values)
test_out.insert(1,"y_pred",pd.Series(pred_test,index=X_test.index).astype(str).values)
test_out.to_csv(out_dir/"test_pred.tsv",sep="\t")

print("Done. Outputs in:",out_dir)

Test metrics:
    Accuracy       AUC    Recall     Prec.        F1     Kappa       MCC  \
0  0.978041  0.999124  0.978041  0.978345  0.978092  0.971909  0.971938   

   TT (Sec)   bal_acc  f1_macro  
0  0.889845  0.964857  0.962208  
Saved model to: mincancer1_Final_tabpfn2.5_cv5_out/final_model.joblib
✅ Done. Outputs in: mincancer1_Final_tabpfn2.5_cv5_out


| Parameter                | Value                                    |
| ------------------------ | ---------------------------------------- |
| Model                    | TabPFNClassifier                         |
| Model version            | tabpfn-v2.5                              |
| Model checkpoint         | tabpfn-v2.5-classifier-v2.5_default.ckpt |
| Random seed              | 2026                                     |
| Missing value imputation | median                                   |
| Cross-validation         | 5-fold                                   |
| Split strategy           | predefined sample indices                |

